# RAG Pipeline: Question Answering over a PDF

A small, end-to-end **Retrieval-Augmented Generation (RAG)** pipeline with built-in evaluation.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Varnika0804/rag-pipeline/blob/main/RAG_Pipeline.ipynb)

**Pipeline**

```
PDF -> clean & chunk -> embed -> vector store -> retrieve top-k -> Gemini answers -> LLM judge scores
```

| Stage | Tool |
|---|---|
| Loading | `PyPDFLoader` |
| Chunking | `RecursiveCharacterTextSplitter` (500 characters, 50 overlap) |
| Embeddings | `sentence-transformers/all-MiniLM-L6-v2` |
| Vector store | Chroma (in-memory) |
| Generation and judging | Google Gemini via `langchain-google-genai` |

**Evaluation metrics** (scored 0-5 by an LLM judge): context relevance, groundedness, answer relevance.

The notebook was developed against one chapter of an A/B testing book, so the sample questions are about experimentation. Swap in your own PDF and questions.

## 1. Setup

Skip the install cell if you already ran `pip install -r requirements.txt`.

In [ ]:
%pip install -q langchain-community langchain-text-splitters langchain-huggingface langchain-chroma langchain-google-genai pypdf sentence-transformers pandas python-dotenv

In [ ]:
import getpass
import os
import re
import time
from pathlib import Path

import pandas as pd

# ---- Configuration ----------------------------------------------------------
PDF_PATH = Path("data/document.pdf")   # put your PDF here, or upload it when prompted
CHUNK_SIZE = 500
CHUNK_OVERLAP = 50
MIN_UNIQUE_WORDS = 15                  # drop chunks with fewer unique words (boilerplate)
EMBEDDING_MODEL = "all-MiniLM-L6-v2"
COLLECTION_NAME = "rag_pipeline"
TOP_K = 10                             # chunks passed to the LLM
LLM_MODEL = os.getenv("GEMINI_MODEL", "gemini-3.5-flash")
MIN_SECONDS_BETWEEN_CALLS = 13        # free tier allows ~5 requests/min; set to 0 on a paid plan
MAX_RETRIES = 5                        # retries when the API says "rate limit exceeded"

### API key

Never paste a key into a notebook cell: it ends up in the file, in Git history, and on GitHub.

- **Colab:** open the key icon in the left sidebar (Secrets), add `GOOGLE_API_KEY`, and enable notebook access.
- **Local:** copy `.env.example` to `.env` and put the key there (`.env` is git-ignored).
- Otherwise the cell below prompts for it without echoing it.

In [ ]:
try:
    from dotenv import load_dotenv
    load_dotenv()                      # reads a local .env file if present
except ImportError:
    pass


def get_google_api_key() -> str:
    """Look for the key in env vars, then Colab Secrets, then prompt for it."""
    key = os.environ.get("GOOGLE_API_KEY")
    if key:
        return key
    try:
        from google.colab import userdata
        return userdata.get("GOOGLE_API_KEY")
    except Exception:
        return getpass.getpass("Enter your Google API key: ")


os.environ["GOOGLE_API_KEY"] = get_google_api_key()

## 2. Load the PDF

`PyPDFLoader` returns one document per page, with the page number in the metadata.

In [ ]:
from langchain_community.document_loaders import PyPDFLoader

if not PDF_PATH.exists():
    try:                               # Colab: ask for an upload
        from google.colab import files
        uploaded = files.upload()
        if not uploaded:
            raise FileNotFoundError("No file was uploaded.")
        PDF_PATH = Path(next(iter(uploaded)))
    except ImportError:
        raise FileNotFoundError(
            f"PDF not found at {PDF_PATH}. Place your PDF there and re-run."
        )

docs = PyPDFLoader(str(PDF_PATH)).load()
print(f"Loaded {len(docs)} pages from {PDF_PATH.name}")
print(f"Page 1 preview: {docs[0].page_content[:200]!r}")

## 3. Clean and chunk

PDFs often repeat a header or footer on every page. In an earlier run of this notebook, a vague query returned the same copyright footer in 8 of the top 10 results, which crowded out real content. `remove_repeated_lines` strips lines that repeat across most pages. Near-empty and duplicate chunks are dropped as well.

In [ ]:
from collections import Counter

from langchain_text_splitters import RecursiveCharacterTextSplitter


def remove_repeated_lines(pages, threshold=0.5, min_pages=5):
    """Remove header/footer lines that appear on at least `threshold` of the pages."""
    if len(pages) < min_pages:         # too few pages to tell boilerplate from content
        return pages, set()

    line_counts = Counter()
    for page in pages:
        line_counts.update({ln.strip() for ln in page.page_content.splitlines() if ln.strip()})

    cutoff = threshold * len(pages)
    boilerplate = {ln for ln, count in line_counts.items() if count >= cutoff}

    for page in pages:
        kept = [ln for ln in page.page_content.splitlines() if ln.strip() not in boilerplate]
        page.page_content = "\n".join(kept)
    return pages, boilerplate


def is_useful(chunk) -> bool:
    return len(set(chunk.page_content.lower().split())) >= MIN_UNIQUE_WORDS


docs, boilerplate = remove_repeated_lines(docs)
print(f"Removed {len(boilerplate)} repeated header/footer lines")

splitter = RecursiveCharacterTextSplitter(chunk_size=CHUNK_SIZE, chunk_overlap=CHUNK_OVERLAP)
raw_chunks = splitter.split_documents(docs)

seen, chunks = set(), []
for chunk in raw_chunks:
    key = " ".join(chunk.page_content.split())
    if is_useful(chunk) and key not in seen:
        seen.add(key)
        chunks.append(chunk)

assert chunks, "No usable chunks were produced. Check the PDF has extractable text."
print(f"{len(raw_chunks)} chunks -> {len(chunks)} after dropping boilerplate and duplicates")
print(f"\nExample chunk (page {chunks[0].metadata['page'] + 1}):\n{chunks[0].page_content}")

## 4. Embed and index

Each chunk is embedded and stored in Chroma. `build_vectorstore` deletes any existing collection first. Chroma's in-memory collection survives between cells, so calling `Chroma.from_documents` twice in one session stores every chunk twice, and each passage then shows up twice in search results.

In [ ]:
from langchain_chroma import Chroma
from langchain_huggingface import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(model_name=EMBEDDING_MODEL)


def build_vectorstore(documents):
    """Create a fresh collection so re-running this cell never stacks duplicates."""
    try:
        Chroma(collection_name=COLLECTION_NAME, embedding_function=embeddings).delete_collection()
    except Exception:
        pass                           # nothing to delete on the first run
    return Chroma.from_documents(documents, embeddings, collection_name=COLLECTION_NAME)


vectorstore = build_vectorstore(chunks)
print(f"Indexed {len(vectorstore.get()['ids'])} chunks")

## 5. Retrieve

`retrieve` over-fetches, removes identical passages, and returns the top `k`. The optional `min_page` filter restricts the search to later pages (0-indexed), which is useful for skipping a table of contents or front matter that happens to mention your query terms.

In [ ]:
def retrieve(query, k=TOP_K, min_page=None):
    """Return the top-k distinct chunks for a query.

    min_page: only search pages >= min_page (0-indexed), e.g. to skip front matter.
    """
    kwargs = {"filter": {"page": {"$gte": min_page}}} if min_page is not None else {}
    candidates = vectorstore.similarity_search(query, k=k * 2, **kwargs)

    seen, results = set(), []
    for doc in candidates:
        key = " ".join(doc.page_content.split())
        if key not in seen:
            seen.add(key)
            results.append(doc)
        if len(results) == k:
            break
    return results


for doc in retrieve("What is a guardrail metric?", k=3):
    print(f"[page {doc.metadata['page'] + 1}] {doc.page_content[:200]!r}\n")

## 6. Generate answers

The prompt restricts Gemini to the retrieved context and tells it to say so when the answer is not there.\n\n`ask_llm` spaces out the calls and retries on rate-limit errors, because the free Gemini tier allows only about 5 requests per minute. The batch evaluation below therefore takes a few minutes on the free tier.

In [ ]:
from langchain_google_genai import ChatGoogleGenerativeAI

llm = ChatGoogleGenerativeAI(model=LLM_MODEL)

PROMPT_TEMPLATE = """Answer the question based ONLY on the following context.
If the context does not contain the answer, say you don't know.

Context:
{context}

Question: {question}
Answer:"""


_last_call = [0.0]


def ask_llm(prompt):
    """Call the LLM, pausing between calls and retrying if the rate limit is hit."""
    for attempt in range(MAX_RETRIES):
        wait = MIN_SECONDS_BETWEEN_CALLS - (time.time() - _last_call[0])
        if wait > 0:
            time.sleep(wait)
        try:
            _last_call[0] = time.time()
            return llm.invoke(prompt)
        except Exception as err:
            rate_limited = "429" in str(err) or "RESOURCE_EXHAUSTED" in str(err)
            if not rate_limited or attempt == MAX_RETRIES - 1:
                raise
            print(f"Rate limit hit, waiting 20s (retry {attempt + 1}/{MAX_RETRIES - 1})...")
            time.sleep(20)


def response_text(response) -> str:
    """Gemini may return a plain string or a list of content blocks; handle both."""
    content = response.content
    if isinstance(content, str):
        return content
    parts = []
    for block in content:
        parts.append(block.get("text", "") if isinstance(block, dict) else str(block))
    return "".join(parts)


def answer_question(question, k=TOP_K, min_page=None):
    docs = retrieve(question, k=k, min_page=min_page)
    context = "\n\n".join(doc.page_content for doc in docs)
    prompt = PROMPT_TEMPLATE.format(context=context, question=question)
    return {
        "question": question,
        "answer": response_text(ask_llm(prompt)),
        "context": context,
        "sources": sorted({doc.metadata["page"] + 1 for doc in docs}),
    }

## 7. Evaluate with an LLM judge

A second Gemini call scores each answer from 0 to 5 on three metrics:

| Metric | Question it answers |
|---|---|
| Context relevance | Did retrieval find passages relevant to the question? |
| Groundedness | Is the answer supported by those passages? |
| Answer relevance | Does the answer actually address the question? |

Caveat: the judge is the same model that wrote the answer, so treat the scores as a rough signal, not a benchmark.

In [ ]:
EVAL_TEMPLATE = """You are an evaluator. Score each metric from 0-5.

Question: {question}
Retrieved Context: {context}
Generated Answer: {answer}

Score these three metrics:
1. Context Relevance: Is the retrieved context relevant to the question? (0-5)
2. Groundedness: Is the answer supported by the context? (0-5)
3. Answer Relevance: Does the answer actually address the question? (0-5)

Respond ONLY in this format:
Context Relevance: X
Groundedness: X
Answer Relevance: X
Explanation: one line why"""

METRICS = ["Context Relevance", "Groundedness", "Answer Relevance"]


def evaluate_rag(result):
    """Score one answer_question() result with an LLM judge and return a dict."""
    text = response_text(ask_llm(EVAL_TEMPLATE.format(**result)))
    scores = {}
    for metric in METRICS:
        match = re.search(rf"{metric}:\s*(\d)", text)
        scores[metric] = int(match.group(1)) if match else None
    explanation = re.search(r"Explanation:\s*(.+)", text)
    scores["Explanation"] = explanation.group(1).strip() if explanation else text.strip()
    return scores

## 8. Run it

First one question end to end, then a small batch.

In [ ]:
result = answer_question("What is a guardrail metric?")
print("ANSWER:", result["answer"])
print("SOURCE PAGES:", result["sources"])
print("\nSCORES:", evaluate_rag(result))

In [ ]:
from IPython.display import display

# Questions written for the sample A/B testing chapter. Replace them for your own PDF.
TEST_QUESTIONS = [
    "What is a guardrail metric?",
    "What is an Overall Evaluation Criterion (OEC)?",
    "Why is randomization important in controlled experiments?",
    "What is Twyman's Law?",
    "What is the capital of France?",   # out of scope: the model should say it doesn't know
]

rows = []
for question in TEST_QUESTIONS:
    result = answer_question(question)
    rows.append({"Question": question, **evaluate_rag(result)})

report = pd.DataFrame(rows)
report[METRICS] = report[METRICS].apply(pd.to_numeric, errors="coerce")
display(report)
print("\nAverage scores:")
print(report[METRICS].mean().round(2))

## Limitations and next steps

- **Chunking:** fixed 500-character chunks can split a definition in half. Try larger chunks, more overlap, or semantic chunking.
- **Retrieval:** pure vector search. Hybrid search (BM25 plus embeddings) or a reranker usually helps on exact-term questions.
- **Evaluation:** five questions and a same-model judge is a smoke test. Use a hand-labelled question set and a different judge model for real comparisons.
- **Persistence:** Chroma runs in memory here. Pass `persist_directory=` to keep the index between sessions.